In [1]:
from functools import partial

import astroplan as ap
import astropy.units as u

from astropaul.database import database_connection, html_path
import astropaul.targetlistcreator as tlc
import astropaul.html as html
import astropaul.phase as ph
import astropaul.priority as pr

%load_ext autoreload
%autoreload 2


In [3]:
name = "APO Observing 2026-Q4"
html_dir = html_path() / name
html.clear_directory(html_dir)

session = tlc.ObservingSession(ap.Observer.at_site("APO"))

session.add_full_day("2026-10-19")
session.add_full_day("2026-10-20")
session.add_full_day("2026-10-22")
session.add_full_day("2026-10-23")
session.add_half_day("2026-10-24", first_half=True)

synthetic_phase_percent = 0.02

phase_event_defs = [
    ph.PhaseEventDef("Mid Eclipse", ph.calc_mid_eclipse),
    ph.PhaseEventDef("Eclipse", ph.calc_mid_eclipse),
    ph.PhaseEventDef(
        "Not in Eclipse", partial(ph.calc_time_of_gress, synthetic_phase_percent=synthetic_phase_percent, ingress=False)
    ),
    ph.PhaseEventDef("Eclipse", partial(ph.calc_time_of_gress, synthetic_phase_percent=synthetic_phase_percent, ingress=True)),
]

min_altitude = 35 * u.deg
max_magnitude = 13

all_targets = tlc.TargetList.load()

creator = tlc.TargetListCreator(name=name, phase_event_defs=phase_event_defs)
creator.steps = [
    partial(tlc.filter_targets, criteria=lambda df: df["List Kostov EBs"]),
    partial(tlc.filter_targets, criteria=lambda df: df["Vmag"] < max_magnitude),
    partial(
        tlc.add_observability,
        observing_session=session,
        calc_moon_distance=True,
        observability_threshold=(min_altitude, 80 * u.deg),
    ),
    partial(tlc.filter_targets, criteria=lambda df: df["Observable Any Night"]),
    partial(
        tlc.add_phase_events,
        observing_session=session,
        phase_event_defs=phase_event_defs,
        event_types=["Mid Eclipse", "Eclipse"],
    ),
]
tl = creator.calculate(initial_list=all_targets, verbose=False)
tl.name = name

print(tl.summarize())

Name: APO Observing 2026-Q4
Criteria:
  Target list loaded from file: TargetList_2026-09-05_12h41m38s.tl (750 targets)
  lambda df: df["List Kostov EBs"] (199 targets)
  lambda df: df["Vmag"] < 13 (111 targets)
  Observability calculated at APO in 15.0 min intervals from 2026-10-19 to 2026-10-24
    AltitudeConstraint: {'min': np.float64(35.0), 'max': np.float64(80.0), 'boolean_constraint': True}
  lambda df: df["Observable Any Night"] (66 targets)
  
66 targets:
    66 QuadEB
Column Count (primary, secondary):
    Target: (3, 4)
    RV Calibration Targets: (1, 2)
    List: (0, 20)
    Count: (8, 0)
    TESS Data: (4, 0)
    Gaia Bailer Jones: (1, 2)
    Observable: (5, 20)
Associated tables:
    3258 rows,   3 columns: Catalog Membership
    1188 rows,   2 columns: List Memberships
     894 rows,   7 columns: Ephemerides
     716 rows, 126 columns: TESS
     318 rows, 105 columns: Gaia DR3
      49 rows,  16 columns: WDS
     511 rows,  12 columns: DSSI Observations
      44 rows,   7

In [5]:
html.clear_directory(html_dir)

altitude_categories = [
    ((-90, min_altitude.value), 0),
    ((min_altitude.value, 90), 1),
]

pl = pr.PriorityList(tl, session, interval=30 * u.min)
pr.calculate_altitude_priority(pl, altitude_categories=altitude_categories)
pr.prioritize_phase_sequence(pl, ["Eclipse"], "Eclipse", True, True, True)
# pr.prioritize_side_observation(pl, side_state="Eclipse")
pr.calculate_overall_priority(pl)
pr.aggregate_target_priorities(pl, skip_column_threshold=0.01)
pl.categorize_priorities(bins=[0.00, 0.20, 0.40, 0.6, 1.00], labels=["", "*", "* *", "* * *"])

html.render_observing_pages(tl, pl, {}, html_dir, target_pages="None")
# pl.categorical_priorities[0]